# MFA-Modell für Roggenstroh - Modulare Version

## Changelog / Änderungsprotokoll

* **Version:** 1.0
* **Datum:** 21.06.2025
* **Autor:** [Johannes Scholz, Lukas Hoppe]
* **Änderungen:**
    * Grundstruktur für modulares Notebook erstellt.
    * Code in Funktionsblöcke unterteilt (Setup, Berechnungen, Visualisierung).

# Section 0: Importing the packages & basic settings translator

In [1]:
### Load packages ###

# Load general libraries
import sys, os
import numpy as np
import pandas as pd
from scipy.stats import lognorm
import xlsxwriter
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator,
                               FormatStrFormatter,
                               AutoMinorLocator)
import warnings
import re
from collections import defaultdict
from scipy.optimize import minimize
import copy

# Load ODYM package
# Add ODYM module directory to system path, absolute
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())),'framework', 'ODYM-master_20241127', 'odym', 'modules')) 

# Import the ODYM class file
import ODYM_Classes as msc 
# Import the ODYM function file
import ODYM_Functions as msf
# Import the dynamic stock model library
import dynamic_stock_model as dsm 


# Load bioDYM_addon
# Add ODYM module directory to system path, absolute
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'framework', 'bioDYM_add-on', 'modules')) 

# Import classes for first order model process
import bioDYM_classes as bicl
# Import plotting functions
import bioDYM_plotting as bipl
# Import export functions
import bioDYM_export as bix




# Enables plotting directly in the notebook (in most cases this is already enabled)
%matplotlib inline

# Section 1: Configuration

# Section 2: Function definitions

### Erklärung der Berechnungs-Engine: Der iterative Solver

Die Berechnung eines komplexen Stoffstromsystems, insbesondere mit Lagern, zeitlichen Verzögerungen und Kreisläufen, erfordert eine robuste Methode, die sicherstellt, dass alle Flüsse in der korrekten Reihenfolge berechnet werden.

#### Das Problem: Das "Wasserfall-Prinzip"

Ein einfacher, rein sequenzieller Ansatz (erst alle TCs, dann alle DSMs, etc.) funktioniert wie ein Wasserfall – die Berechnung fließt nur in eine Richtung. Dies scheitert, wenn es im System Kreisläufe oder komplexe Abhängigkeiten gibt. Ein Prozess `C` muss dann vielleicht auf das Ergebnis eines Prozesses `B` warten, der seinerseits auf `A` wartet. Dieser starre Ablauf kann zu Berechnungsfehlern oder "stillen Fehlern" führen, bei denen falsche Zwischenergebnisse verwendet werden.

#### Die Lösung: Ein "iterativer Solver"

Die hier implementierte Methode ist ein **iterativer Solver**. Anstatt zu versuchen, die "richtige" Reihenfolge vorab zu erraten, dreht die Rechen-Engine so lange Runden ("Iterationen"), bis sich im gesamten System nichts mehr ändert und es einen stabilen Zustand erreicht hat.

**Der Ablauf in jeder einzelnen Iteration:**

1.  **Berechnung der "einfachen" TC-Flüsse:** Der Solver versucht zuerst, alle normalen, TC-basierten Flüsse zu berechnen. **Wichtig:** Er überspringt dabei aber konsequent alle Flüsse, deren Startprozess als "Spezialprozess" (DSM oder FOMP) definiert ist. Damit wird verhindert, dass die Ergebnisse der Spezial-Logik vorzeitig mit einer falschen TC-Berechnung überschrieben werden.

2.  **Ausführung der Spezial-Modelle (DSM/FOMP):** Als Nächstes prüft der Solver für jeden Spezialprozess, ob alle seine Zuflüsse mittlerweile bekannt sind.
    * Wenn ja, wird die entsprechende Spezialfunktion (`calculate_dynamic_stock` oder `calculate_fomp`) ausgeführt, die den korrekten, physikalisch basierten Abfluss berechnet.
    * Wenn nein, wartet der Solver bis zur nächsten Runde.

3.  **Wiederholung und Konvergenz:** Dieser gesamte Prozess (Schritt 1 & 2) wird wiederholt. In der nächsten Runde sind die Abflüsse der Spezialmodelle aus der vorherigen Runde nun bekannte Zuflüsse für andere Prozesse. Dadurch können in der TC-Berechnung weitere, bisher unlösbare Flüsse berechnet werden. Die Information "frisst" sich so lange durch das System, bis alle Flüsse bekannt sind.

4.  **Stabilitäts-Check:** Wenn die Engine eine komplette Runde durchläuft, ohne einen einzigen neuen Wert berechnen zu können, bedeutet das, das System ist stabil (konvergiert). Die Schleife bricht dann ab, um unnötige Rechenzeit zu sparen.

Dieser iterative Ansatz ist die Standardmethode zur Lösung komplexer Stoffstrom-Systeme, da er Abhängigkeiten und Kreisläufe robust und automatisch auflöst.

In [4]:
# TEMPORÄRE DEBUG-ZELLE: Spaltennamen überprüfen
import pandas as pd

# Bitte stellen Sie sicher, dass der Dateiname korrekt ist
excel_file_path = '250625_Template_CS0.xlsx' 

df_check = pd.read_excel(excel_file_path, sheet_name='2_3_Process_TCs')

print("Gefundene Spalten im Blatt '2_3_Process_TCs':")
print(list(df_check.columns))

Gefundene Spalten im Blatt '2_3_Process_TCs':
['Spalte1', 'ID', 'Process_ID', 'Name(EN)', 'Region', 'Carbon_Stock', 'Life_Phase', 'Description', 'Process_Type', 'TC?', 'Dyn_TC?', 'Stock?', 'Initial_Stock?', 'DSM?', 'FOMP?', 'Nr. Outflows?', 'Output_Flow', 'Flow_ID', 'TC_ID', 'TC_Value', 'Titel', 'Year publication', 'Author', 'Type of the Study', 'URL']


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


### 2.1: Define Model Scope & Classifications

In [5]:
# ====================================================================
# Section 2: Function Definitions
# ====================================================================

# --- Helper Function 2.1: Define Model Scope & Classifications ---
def define_model_scope(start_year, end_year, elements):
    """
    Defines the temporal and elemental scope of the MFA model.

    Args:
        start_year (int): The first year of the analysis.
        end_year (int): The last year of the analysis.
        elements (list): A list of strings for the elements to be tracked.

    Returns:
        tuple: A tuple containing the ModelClassification dictionary 
               and the IndexTable DataFrame, which are core ODYM objects.
    """
    ModelClassification = {}
    MyYears = list(np.arange(start_year, end_year + 1))
    
    # Define Time and Element Classifications for the ODYM framework
    ModelClassification['Time'] = msc.Classification(Name='Time', Dimension='Time', ID=1, Items=MyYears)
    ModelClassification['Element'] = msc.Classification(Name='Elements', Dimension='Element', ID=2, Items=elements)

    # Create the IndexTable, which ODYM uses for calculations
    IndexTable = pd.DataFrame({
        'Aspect': ['Time', 'Element'],
        'Description': ['Model aspect "time"', 'Model aspect "Element"'],
        'Dimension': ['Time', 'Element'],
        'Classification': [ModelClassification[Aspect] for Aspect in ['Time', 'Element']],
        'IndexLetter': ['t', 'e']
    })
    IndexTable.set_index('Aspect', inplace=True)
    
    print("--> Model scope and classifications defined.")
    return ModelClassification, IndexTable

### 2.2: Initialize the main MFA System object

In [6]:
# --- Helper Function 2.2: Initialize the main MFA System object ---
def initialize_mfa_system(model_classification, index_table):
    """
    Initializes the main MFAsystem object based on the defined scope.

    Args:
        model_classification (dict): The ModelClassification dictionary from define_model_scope.
        index_table (pd.DataFrame): The IndexTable DataFrame from define_model_scope.

    Returns:
        odym.MFAsystem: An empty but structured MFAsystem object.
    """
    # Extract scope details from the input objects
    start_time = model_classification['Time'].Items[0]
    end_time = model_classification['Time'].Items[-1]
    element_items = model_classification['Element'].Items
    
    # Create the main system object using the ODYM class
    MFA_System = msc.MFAsystem(
        Name='RyeStrawMFA', 
        Geogr_Scope='Case_Study_Region', 
        Unit='Mg', 
        ProcessList=[], 
        FlowDict={}, 
        StockDict={},
        ParameterDict={}, 
        Time_Start=start_time, 
        Time_End=end_time, 
        IndexTable=index_table, 
        Elements=element_items
    )
    
    print("--> MFA system object initialized.")
    return MFA_System

### 2.3 validate_input_data

In [7]:
# ====================================================================
# NEUE FUNKTION ZUR DATENVALIDIERUNG
# ====================================================================
def validate_input_data(excel_data_dict):
    """
    Prüft, ob die geladenen Excel-Daten die erwartete Struktur haben.
    Wirft einen ValueError mit einer klaren Fehlermeldung, wenn etwas fehlt.
    """
    print("--> Validating input data structure...")

    # Definition der erwarteten minimalen Struktur
    # Format: { 'Blattname': ['erwartete_spalte_1', 'erwartete_spalte_2', ...] }
    REQUIRED_STRUCTURE = {
        '1_1_Definition_Flows': ['Flow_ID', 'Name(EN)', 'Process_ID_O', 'Process_ID_I'],
        '1_2_Data_Flows': ['Flow_ID', 'Year_Flow', 'Flow_Py'],
        '2_1_Definition_Processes': ['ID', 'Name(EN)', 'Stock?', 'Initial_Stock?'],
        '2_4_Process_Stock_': ['Process_ID', 'Initial_Stock_material'],
        '2_5_dynamic_tcs': ['TC_ID', 'Year', 'Value']
    }

    for sheet_name, required_columns in REQUIRED_STRUCTURE.items():
        # 1. Prüfen, ob das Tabellenblatt existiert
        if sheet_name not in excel_data_dict:
            raise ValueError(f"FEHLER: Das erforderliche Tabellenblatt '{sheet_name}' wurde in der Excel-Datei nicht gefunden!")

        # 2. Prüfen, ob alle erforderlichen Spalten im Blatt vorhanden sind
        existing_columns = excel_data_dict[sheet_name].columns
        for col in required_columns:
            if col not in existing_columns:
                raise ValueError(f"FEHLER: Im Tabellenblatt '{sheet_name}' fehlt die erforderliche Spalte '{col}'!")

    print("--> Input data validation successful. All required sheets and columns are present.")

### 2.3 load_and_define_processes

In [8]:
# ===================================================================
# FINAL CORRECTED FUNCTION
# ===================================================================
def load_and_define_processes(mfa_system, excel_path):
    """
    This function now ONLY defines the structure of processes and stocks,
    without setting any initial values.
    """
    print("--> Defining process and stock structures...")
    
    input_data = pd.read_excel(excel_path, sheet_name=None, header=0, engine='openpyxl', na_values=['N.A.', 'NA', 'n/a'])
    validate_input_data(input_data) # Validation remains important

    process_definitions = input_data['2_1_Definition_Processes']
    for index, row in process_definitions.iterrows():
        if pd.notna(row['Name(EN)']):
            process_id = int(row['ID'])
            has_tcs = 'TC' if 'TC?' in row and row['TC?'] == 'Yes' else 'None'
            mfa_system.ProcessList.append(msc.Process(Name=row['Name(EN)'], ID=process_id, Extensions=has_tcs))
            
            # Create stock objects if needed
            if 'Stock?' in row and row['Stock?'] == 'Yes':
                mfa_system.StockDict[f"dS_{process_id}"] = msc.Stock(Name=f"dS_{process_id}", P_Res=process_id, Type=1, Indices='t,e')
                mfa_system.StockDict[f"S_{process_id}"] = msc.Stock(Name=f"S_{process_id}", P_Res=process_id, Type=0, Indices='t,e')

    # Values will be set in the next function
    return mfa_system, input_data

In [9]:
# ===================================================================
# NEUE KORRIGIERTE FUNKTION zum Laden der DSM-Parameter
# ===================================================================

def load_dsm_parameters(excel_data):
    """
    Liest das Blatt '3_1_Definition_DSM' und erstellt das DSM_PARAMS Dictionary.
    
    BUGFIX: Diese Version wandelt die Spalte 'Process_ID' explizit in den
    Datentyp Integer um. Dies behebt den Fehler, bei dem Prozess-IDs als
    Float (z.B. 7.0) statt als Integer (z.B. 7) eingelesen wurden, was zu
    fehlgeschlagenen Zugriffen auf die Lagerobjekte führte.
    """
    sheet_name = '3_1_Definition_DSM'
    print(f"--> Loading DSM parameters from sheet '{sheet_name}'...")
    
    if sheet_name not in excel_data:
        print(f"--> INFO: Sheet '{sheet_name}' not found. Using empty DSM configuration.")
        return {}

    df_dsm = excel_data[sheet_name]
    
    # --- START DES BUGFIX ---
    # Prüfen, ob die Spalte existiert, um Folgefehler zu vermeiden.
    if 'Process_ID' not in df_dsm.columns:
        print(f"--> FATAL ERROR: Spalte 'Process_ID' nicht im Blatt '{sheet_name}' gefunden.")
        return {}
        
    # Entferne Zeilen ohne Prozess_ID und erzwinge den Datentyp Integer.
    # Dies ist der entscheidende Schritt, um den 7 vs 7.0 Fehler zu beheben.
    df_dsm = df_dsm.dropna(subset=['Process_ID'])
    df_dsm['Process_ID'] = df_dsm['Process_ID'].astype(int)
    # --- ENDE DES BUGFIX ---
    
    dsm_params = {}

    # Gruppiere nach der nun korrekten Integer-ID
    for process_id, group in df_dsm.groupby('Process_ID'):
        # 'process_id' ist jetzt garantiert ein Integer
        group = group.sort_values(by='Category_ID')
        
        dsm_params[process_id] = {
            'inflow_split': list(group['Inflow_Split_[%]']),
            'lifetimes': {
                'Type': list(group['Lifetime_Type'])[0],
                'Mean': list(group['Lifetime_Mean']),
                'StdDev': list(group['Lifetime_StdDev'])
            },
            'category_names': list(group['Category_Name'])
        }
        
    print(f"--> Successfully loaded configurations for {len(dsm_params)} DSM process(es).")
    return dsm_params

In [10]:
def load_fomp_parameters(excel_data):
    """
    Reads the '3_2_Definition_FOMP' sheet and constructs the FOMP_PARAMS dictionary.
    This version ignores empty rows.
    """
    # ÄNDERUNG: Korrekter Blattname
    sheet_name = '3_2_Definition_FOMP' 
    print(f"--> Loading FOMP parameters from sheet '{sheet_name}'...")

    if sheet_name not in excel_data:
        print(f"--> INFO: Sheet '{sheet_name}' not found. Using empty FOMP configuration.")
        return {}
        
    df_fomp = excel_data[sheet_name]
    fomp_params = {}

    for _, row in df_fomp.iterrows():
        # ÄNDERUNG: Prüfen, ob die Zeile leer ist, bevor wir sie verarbeiten
        if pd.isna(row['Process_ID']):
            continue # Überspringe diese Zeile und gehe zur nächsten

        process_id = int(row['Process_ID'])
        param_name = row['Parameter_Name']
        value = row['Value']
        
        if process_id not in fomp_params:
            fomp_params[process_id] = {}
        
        try:
            fomp_params[process_id][param_name] = float(value)
        except (ValueError, TypeError):
            fomp_params[process_id][param_name] = value
            
    print(f"--> Successfully loaded configurations for {len(fomp_params)} FOMP process(es).")
    return fomp_params

In [11]:
# ===================================================================
# ERSETZTE LADEFUNKTION FÜR UNSICHERHEITEN
# ===================================================================
def load_uncertainty_definitions(excel_data):
    """
    Reads the '3_Uncertainty_Parameters' sheet with the new, self-documenting
    structure and converts it into the UNCERTAINTY_PARAMS dictionary format.
    """
    sheet_name = '4_1_Uncertainty_Parameters'
    print(f"--> Loading uncertainty definitions from sheet '{sheet_name}'...")
    
    if sheet_name not in excel_data:
        print(f"--> INFO: Sheet '{sheet_name}' not found. No uncertainties will be loaded.")
        return {}

    df_uncertainty = excel_data[sheet_name].dropna(subset=['Parameter_Name'])
    uncertainty_params = {}

    for _, row in df_uncertainty.iterrows():
        param_name = row['Parameter_Name']
        dist_type = row['Distribution']
        definition = {'distribution': dist_type}
        
        # Read values only if they are not empty (NaN)
        if dist_type == 'uniform':
            if pd.notna(row['Min']) and pd.notna(row['Max']):
                definition['min'] = row['Min']
                definition['max'] = row['Max']
        elif dist_type == 'normal':
            if pd.notna(row['Mean']) and pd.notna(row['StdDev']):
                definition['mean'] = row['Mean']
                definition['std'] = row['StdDev']
        elif dist_type == 'triangular':
            if pd.notna(row['Min']) and pd.notna(row['Mode']) and pd.notna(row['Max']):
                definition['min'] = row['Min']
                definition['mode'] = row['Mode']
                definition['max'] = row['Max']
        
        if len(definition) > 1: # Add only if parameters were found
            uncertainty_params[param_name] = definition
        
    print(f"--> Successfully loaded {len(uncertainty_params)} uncertainty parameter definition(s).")
    return uncertainty_params

### 2.4: Define flows and all model parameters

In [12]:
# ===================================================================
# KORREKTUR der Funktion define_flows_and_parameters (für MC)
# ===================================================================
def define_flows_and_parameters(mfa_system, all_excel_data, dsm_params_config, fomp_params_config, tc_updates=None):
    """
    Diese Funktion definiert Flüsse, initialisiert ALLE Systemwerte, befüllt sie
    UND definiert alle Modellparameter (TCs, Inhalte etc.).
    
    NEU: Sie akzeptiert ein optionales 'tc_updates'-Dictionary, um die für einen
    Monte-Carlo-Lauf gesampelten Transferkoeffizienten anzuwenden.
    """
    print("--> Defining flows, parameters, and setting all initial values...")
    
    # Schritt 1 & 2: Strukturen definieren und alles auf Null initialisieren
    flow_definitions = all_excel_data['1_1_Definition_Flows']
    for _, row in flow_definitions.iterrows():
        if pd.notna(row['Name(EN)']):
            start_id, end_id = int(row['Process_ID_O']), int(row['Process_ID_I'])
            mfa_system.FlowDict[row['Flow_ID']] = msc.Flow(Name=row['Flow_ID'], P_Start=start_id, P_End=end_id, Indices='t,e')
    mfa_system.Initialize_StockValues()
    mfa_system.Initialize_FlowValues()
    print("--> All stocks and flows initialized to zero.")

    # Schritt 3: Alle bekannten Werte setzen (Input-Flüsse & Initial Stocks)
    flow_data = all_excel_data['1_2_Data_Flows']
    for flow_id, flow_obj in mfa_system.FlowDict.items():
        if flow_id in flow_data['Flow_ID'].values:
            flow_time_series = flow_data[flow_data['Flow_ID'] == flow_id]
            if len(flow_time_series) == len(mfa_system.IndexTable.Classification['Time'].Items):
                flow_obj.Values[:, 0] = np.array(flow_time_series['Flow_Py']).ravel()
    print("--> Populated data for primary input flows.")
    
    initial_stock_data = all_excel_data.get('2_4_Process_Stock_')
    process_definitions = all_excel_data['2_1_Definition_Processes']
    if initial_stock_data is not None:
        for _, row in process_definitions.iterrows():
            if pd.notna(row['ID']) and 'Initial_Stock?' in row and row['Initial_Stock?'] == 'Yes':
                process_id = int(row['ID'])
                stock_data_row = initial_stock_data[initial_stock_data['Process_ID'] == process_id]
                if not stock_data_row.empty:
                    stock_s = mfa_system.StockDict.get(f"S_{process_id}")
                    if stock_s:
                        mat = stock_data_row['Initial_Stock_material'].iloc[0]
                        wc_p = stock_data_row['Initial_Stock_WC[%]'].iloc[0]
                        dm_p = stock_data_row['Initial_Stock_DM[%]'].iloc[0]
                        cc_p = stock_data_row['Initial_Stock_CC[%]'].iloc[0]
                        stock_s.Values[0, :] = [mat, mat * wc_p, mat * dm_p, mat * cc_p]
    
    # Schritt 4: ALLE Standard-Parameter definieren
    parameter_id_counter = 1
    tc_definitions = all_excel_data['2_3_Process_TCs']
    for _, row in tc_definitions.iterrows():
        if 'TC_ID' in row and pd.notna(row['TC_ID']) and pd.notna(row['TC_Value']):
            mfa_system.ParameterDict[row['TC_ID']] = msc.Parameter(Name=row['TC_ID'], ID=parameter_id_counter, Values=row['TC_Value'], Unit='1')
            parameter_id_counter += 1
            
    dynamic_tc_sheet = all_excel_data.get('2_5_dynamic_tcs')
    if dynamic_tc_sheet is not None:
        dynamic_tcs = create_dynamic_tc_parameters(dynamic_tc_sheet, mfa_system.IndexTable.Classification['Time'].Items)
        for name, values in dynamic_tcs.items():
            mfa_system.ParameterDict[name] = msc.Parameter(Name=name, ID=parameter_id_counter, Values=values, Unit='1')
            parameter_id_counter += 1
    
    content_definitions = all_excel_data['1_1_Definition_Flows']
    for _, row in content_definitions.iterrows():
        if pd.notna(row['Flow_ID']) and row['Flow_ID'] in mfa_system.FlowDict:
            for element in ['WC', 'DM', 'CC']:
                if element in row and pd.notna(row[element]):
                    param_name = f"{element}_{row['Flow_ID']}"
                    mfa_system.ParameterDict[param_name] = msc.Parameter(Name=param_name, ID=parameter_id_counter, Values=row[element], Unit='1')
                    parameter_id_counter += 1
    
    # --- START DER MC-ANPASSUNG ---
    # Überschreibe Standard-TCs mit den für diesen MC-Lauf gesampelten Werten
    if tc_updates:
        print("    - Wende Monte-Carlo-Updates auf TC-Parameter an...")
        for param_name, new_value in tc_updates.items():
            if param_name in mfa_system.ParameterDict:
                mfa_system.ParameterDict[param_name].Values = new_value
            else:
                print(f"      - WARNUNG: Gesampelter Parameter '{param_name}' nicht im System-ParameterDict gefunden.")
    # --- ENDE DER MC-ANPASSUNG ---

    print(f"--> Defined {len(mfa_system.ParameterDict)} parameters in total.")

    # Schritt 5: Elementgehalte für primäre Input-Flüsse berechnen
    for flow in mfa_system.FlowDict.values():
        if np.any(flow.Values[:, 0] != 0):
            for i_elem, element_name in enumerate(mfa_system.Elements[1:], 1):
                param_name = f"{element_name}_{flow.Name}"
                if param_name in mfa_system.ParameterDict:
                    content_value = mfa_system.ParameterDict[param_name].Values
                    flow.Values[:, i_elem] = flow.Values[:, 0] * content_value
    
    mfa_system.Consistency_Check()
    
    return mfa_system, all_excel_data

### 2.5 dynamic TC parameters

In [13]:
# Helper Function 2.5: Create dynamic TC time series (More Robust Version)
def create_dynamic_tc_parameters(dynamic_tc_data, time_vector):
    """
    Generates time series for TCs, now with data cleaning to ignore empty rows
    and a pre-flight check to detect and report duplicate entries.
    """
    print("--> Generating dynamic TC time series via interpolation...")

    # --- NEW: Data cleaning step ---
    # Drop all rows where 'TC_ID' or 'Year' are empty, before doing anything else.
    required_cols = ['TC_ID', 'Year', 'Value']
    if not all(col in dynamic_tc_data.columns for col in required_cols):
        print(f"--> FATAL ERROR: The '2_5_dynamic_tcs' sheet is missing one of the required columns: {required_cols}.")
        return {}
    
    cleaned_data = dynamic_tc_data.dropna(subset=['TC_ID', 'Year'])

    # --- Now, run the pre-flight check on the CLEANED data ---
    duplicates = cleaned_data[cleaned_data.duplicated(subset=['TC_ID', 'Year'], keep=False)]
    
    if not duplicates.empty:
        print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print("!!! FATAL ERROR: Duplicate entries found for the same TC in the same year. !!!")
        print("    The following rows in your '2_5_dynamic_tcs' sheet are conflicting:")
        print(duplicates.sort_values(by=['TC_ID', 'Year']))
        print("\n    Please correct the Excel file. Aborting dynamic TC creation.")
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")
        return {}
    
    # --- Continue the rest of the function using 'cleaned_data' ---
    dynamic_tc_dict = {}
    time_array = np.array(time_vector)
    unique_tc_ids = cleaned_data['TC_ID'].unique()
    
    for tc_id in unique_tc_ids:
        tc_points = cleaned_data[cleaned_data['TC_ID'] == tc_id]
        ts = pd.Series(tc_points['Value'].values, index=tc_points['Year'])
        ts_full = ts.reindex(time_vector)
        ts_interpolated = ts_full.interpolate(method='linear', limit_direction='both')
        dynamic_tc_dict[tc_id] = ts_interpolated.to_numpy()

    print(f"--> Generated {len(dynamic_tc_dict)} dynamic TC parameter(s).")
    return dynamic_tc_dict

### 2.6 DSM function

In [14]:
# ===================================================================
# FINALE, SAUBERE VERSION der Funktion calculate_dynamic_stock
# ===================================================================
def calculate_dynamic_stock(mfa_system, dsm_params_config):
    """
    Calculates the outflow from a dynamic stock, correctly handling both
    new inflows and the decay of a non-zero initial stock.
    """
    time_vector = np.array(mfa_system.IndexTable.Classification['Time'].Items)
    num_years, num_elements = len(time_vector), len(mfa_system.Elements)
    dsm_details_results = {} 

    for process_id, params in dsm_params_config.items():
        stock_s = mfa_system.StockDict.get(f"S_{process_id}")
        initial_stock_vector = stock_s.Values[0, :].copy() if stock_s is not None else np.zeros(num_elements)
        inflows = [f.Values for f in mfa_system.FlowDict.values() if f.P_End == process_id]
        total_inflow_values = sum(inflows) if inflows else np.zeros((num_years, num_elements))
        outflow_flow_name = next((f.Name for f in mfa_system.FlowDict.values() if f.P_Start == process_id), None)
        if not outflow_flow_name: continue
        lt_params = params.get('lifetimes', {})
        mean_lifetimes = lt_params.get('Mean', [])
        
        outflow_from_inflows_material, stock_from_inflows_by_cat = np.zeros(num_years), []
        inflow_split, std_devs = params.get('inflow_split', [1.0]), lt_params.get('StdDev', [])
        for i in range(len(inflow_split)):
            inflow_category = total_inflow_values[:, 0] * inflow_split[i]
            dsm_model = dsm.DynamicStockModel(t=time_vector, i=inflow_category, lt={'Type': lt_params.get('Type'), 'Mean': [mean_lifetimes[i]], 'StdDev': [std_devs[i]]})
            s_c, o_c = dsm_model.compute_s_c_inflow_driven(), dsm_model.compute_o_c_from_s_c()
            if o_c is not None:
                outflow_from_inflows_material += o_c.sum(axis=1)
                stock_from_inflows_by_cat.append(s_c.sum(axis=1))
            else: 
                stock_from_inflows_by_cat.append(np.zeros(len(time_vector)))

        avg_lifetime = np.mean(mean_lifetimes) if mean_lifetimes else 0
        decay_rate_k = 1 / avg_lifetime if avg_lifetime > 0 else 0
        outflow_from_initial_stock_ts, decaying_stock_ts = np.zeros_like(total_inflow_values), np.zeros_like(total_inflow_values)

        if np.sum(initial_stock_vector) > 0:
            current_decaying_stock = initial_stock_vector.copy()
            for t in range(num_years):
                decaying_stock_ts[t, :] = current_decaying_stock
                outflow_t = current_decaying_stock * decay_rate_k
                outflow_from_initial_stock_ts[t, :] = outflow_t
                current_decaying_stock -= outflow_t
        
        total_outflow_material = outflow_from_inflows_material + outflow_from_initial_stock_ts[:, 0]
        mfa_system.FlowDict[outflow_flow_name].Values[:, 0] = total_outflow_material
        for elem_idx in range(1, total_inflow_values.shape[1]):
            factor = np.divide(total_inflow_values[:, elem_idx], total_inflow_values[:, 0], out=np.zeros_like(total_inflow_values[:, 0]), where=total_inflow_values[:, 0]!=0)
            mfa_system.FlowDict[outflow_flow_name].Values[:, elem_idx] = total_outflow_material * factor

        dsm_details_results[process_id] = {
            'initial_stock_ts': decaying_stock_ts,
            'inflow_stock_ts_by_cat': stock_from_inflows_by_cat,
            'category_names': params.get('category_names', []),
            'mean_lifetimes': mean_lifetimes
        }
            
    return mfa_system, dsm_details_results

### 2.7 final mass balance

In [15]:
# Helper Function: Calculate final stock balances for ALL processes
def calculate_final_balances(mfa_system):
    # Diese Funktion ist aus Ihrem initialen Notebook und funktionierte korrekt.
    print("--> Calculating final stock balances for ALL processes...")
    num_years = len(mfa_system.IndexTable.Classification['Time'].Items)
    
    for pid in {p.ID for p in mfa_system.ProcessList}:
        if f"S_{pid}" in mfa_system.StockDict:
            stock_s, stock_ds = mfa_system.StockDict[f"S_{pid}"], mfa_system.StockDict[f"dS_{pid}"]
            
            inflows = [f.Values for f in mfa_system.FlowDict.values() if f.P_End == pid]
            outflows = [f.Values for f in mfa_system.FlowDict.values() if f.P_Start == pid]
            total_inflows = sum(inflows) if inflows else np.zeros_like(stock_s.Values)
            total_outflows = sum(outflows) if outflows else np.zeros_like(stock_s.Values)
            dS_values = total_inflows - total_outflows
            stock_ds.Values = dS_values

            initial_stock_vector = stock_s.Values[0, :].copy()
            
            new_s_values = np.zeros_like(stock_s.Values)
            for t in range(num_years):
                stock_t_minus_1 = new_s_values[t-1, :] if t > 0 else initial_stock_vector
                new_s_values[t, :] = stock_t_minus_1 + dS_values[t, :]
            
            stock_s.Values = new_s_values
    
    print("--> Stock balance calculation finished.")
    return mfa_system

### 2.8 FOMP function

### Erklärung: First-Order Model Process (FOMP)

Ein **First-Order Model Process** (Modellprozess erster Ordnung) wird verwendet, um Prozesse zu simulieren, bei denen die Rate des Austrags (z.B. Abbau, Mineralisierung, Emission) direkt von der Menge des im System vorhandenen Materials (dem Lagerbestand) abhängt.

**Analogie:** Man kann es sich wie eine Badewanne mit offenem Abfluss vorstellen. Je mehr Wasser (Lager) in der Wanne ist, desto höher ist der Druck und desto schneller fließt das Wasser ab (Abfluss). Der Abfluss verlangsamt sich, wenn das Lager kleiner wird.

In diesem Modell wird der FOMP verwendet, um die **Mineralisierung von Biokohle im Boden** (Prozess 17, `Lithosphere_Stock`) über die Zeit zu simulieren. Die vereinfachte Formel, die wir verwenden, lautet:

`Mineralisierungsrate(t) = Lager(t-1) * k`

* `Lager(t-1)` ist der Kohlenstoffbestand im Boden aus dem Vorjahr.
* `k` ist die **Abbaurate** oder **Zerfallskonstante**, die angibt, welcher prozentuale Anteil des Lagerbestands pro Jahr abgebaut wird.

Ein kleiner `k`-Wert bedeutet einen langsamen Abbau und eine lange Verweildauer im Boden, was für die Kohlenstoffsequestrierung erwünscht ist.


* Material	Zerfallskonstante k (pro Jahr)	Ungefähre Halbwertszeit	Anmerkung
* Stroh (unbehandelt)	0.1 - 0.5	1.5 - 7 Jahre	Zersetzt sich relativ schnell im Boden.
* Wurzelmasse	0.05 - 0.2	3.5 - 14 Jahre	Etwas stabiler als oberirdisches Stroh.
* Biokohle (labil)	0.01 - 0.05	14 - 70 Jahre	Der Anteil der Biokohle, der sich schneller zersetzt. Dein Wert von 0.032 passt gut in diese Kategorie.
* Biokohle (stabil)	0.0001 - 0.001	700 - 7000 Jahre	Der Großteil der Biokohle ist extrem langlebig und trägt zur langfristigen Kohlenstoffspeicherung bei.


In [16]:
# Helper Function: Calculate FOMP OUTFLOWS
def calculate_fomp(mfa_system, fomp_params_config):
    # Diese Funktion ist aus Ihrem initialen Notebook und funktionierte korrekt.
    print("--> Calculating FOMP outflows...")
    time_vector = mfa_system.IndexTable.Classification['Time'].Items
    num_years, num_elements = len(time_vector), len(mfa_system.Elements)
    
    for process_id, params in fomp_params_config.items():
        if not any(p.ID == process_id for p in mfa_system.ProcessList): continue
        stock_s = mfa_system.StockDict.get(f"S_{process_id}")
        if stock_s is None: continue
        
        print(f"    ... calculating outflow from FOMP process {process_id}")
        initial_stock_vector = stock_s.Values[0, :].copy()
        outflow_flow_name = params.get('outflow_id')
        if not outflow_flow_name: continue
            
        f, k1, k2 = params.get('f', 0), params.get('k1', 0), params.get('k2', 0)
        inflows = [flow.Values for flow in mfa_system.FlowDict.values() if flow.P_End == process_id]
        inflow_values = sum(inflows) if inflows else np.zeros((num_years, num_elements))
        
        new_outflow_values, current_stock = np.zeros_like(inflow_values), initial_stock_vector
        
        for t in range(num_years):
            outflow_t = (inflow_values[t, :] * f) + (current_stock * k1) + (inflow_values[t, :] * k2)
            new_outflow_values[t, :] = outflow_t
            current_stock = current_stock + inflow_values[t, :] - outflow_t

        mfa_system.FlowDict[outflow_flow_name].Values = new_outflow_values

    print("--> FOMP outflow calculation finished.")
    return mfa_system

### 2.9 Plot Mass Balance Error per Process

In [17]:
# --- Helper Function 4.2: Plot Mass Balance Error per Process (New Version) ---
def plot_mass_balance_error(mfa_system_results):
    """
    Creates an interactive bar chart showing the mass balance error for each process.
    Error = Inflows - Outflows - dS. An error of 0 means perfect balance.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, IntSlider, Dropdown

    process_names = [p.Name for p in mfa_system_results.ProcessList]
    time_items = mfa_system_results.IndexTable.Classification['Time'].Items
    element_items = mfa_system_results.Elements
    
    fig = go.FigureWidget()

    def update_plot(year, element):
        year_index = time_items.index(year)
        element_index = element_items.index(element)
        
        errors = []
        for p in mfa_system_results.ProcessList:
            in_val = sum(f.Values[year_index, element_index] for f in mfa_system_results.FlowDict.values() if f.P_End == p.ID)
            out_val = sum(f.Values[year_index, element_index] for f in mfa_system_results.FlowDict.values() if f.P_Start == p.ID)
            ds_val = mfa_system_results.StockDict.get(f'dS_{p.ID}', None)
            ds_sum = ds_val.Values[year_index, element_index] if ds_val is not None else 0
            
            error = in_val - out_val - ds_sum
            errors.append(error)
        
        # Color bars based on error direction
        colors = ['#d62728' if e > 1e-9 else '#2ca02c' if e < -1e-9 else '#7f7f7f' for e in errors] # Red for positive, Green for negative, Grey for zero

        with fig.batch_update():
            fig.data = [] # Clear previous data
            fig.add_trace(go.Bar(x=process_names, y=errors, marker_color=colors))
            fig.update_layout(
                title=f"Mass Balance Error Check for {element.upper()} in {year}",
                yaxis_title="Error in Mg (positive = mass created)",
                shapes=[dict(type='line', y0=0, y1=0, x0=-0.5, x1=len(process_names)-0.5, line=dict(color='black', width=2))] # Zero line
            )

    # Create widgets
    year_slider = IntSlider(min=time_items[0], max=time_items[-1], step=1, value=time_items[0], description='Year')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, year=year_slider, element=element_dropdown)
    display(fig)

### 2.11 sample_parameters

In [18]:
# ===================================================================
# NEW FUNCTION FOR MONTE CARLO SAMPLING
# ===================================================================
def sample_parameters(uncertainty_defs):
    """
    Draws a new random value for each defined uncertain parameter.
    Returns a dictionary with the new values.
    """
    sampled_values = {}
    for param_name, definition in uncertainty_defs.items():
        dist_type = definition.get('distribution')

        if dist_type == 'uniform':
            sampled_values[param_name] = np.random.uniform(definition['min'], definition['max'])
        elif dist_type == 'normal':
            sampled_values[param_name] = np.random.normal(definition['mean'], definition['std'])
        elif dist_type == 'triangular':
            sampled_values[param_name] = np.random.triangular(definition['min'], definition['mode'], definition['max'])
        elif dist_type == 'lognormal':
             sampled_values[param_name] = np.random.lognormal(definition['mean'], definition['std'])
        else:
            print(f"WARNING: Unknown distribution type '{dist_type}' for parameter '{param_name}'. Parameter will not be sampled.")
            
    return sampled_values

## 2.10 Main Calculation Function

In [19]:
# ===================================================================
# KORRIGIERTE Haupt-Berechnungsfunktion (für MC)
# ===================================================================
def run_mfa_calculation(dsm_params, fomp_params, tc_updates=None):
    """
    Diese Funktion ist der iterative Solver.
    
    NEU: Sie akzeptiert ein optionales 'tc_updates'-Dictionary, um 
    Monte-Carlo-Simulationen zu ermöglichen, bei denen TCs für jeden
    Lauf modifiziert werden.
    """
    # Schritt 1: Das System für diesen Lauf von Grund auf neu aufbauen.
    mfa_system, _ = load_and_define_processes(
        initialize_mfa_system(model_classification, index_table),
        EXCEL_FILE_PATH
    )
    # WICHTIG: Reiche die tc_updates an die nächste Funktion durch
    mfa_system, _ = define_flows_and_parameters(mfa_system, all_excel_data, dsm_params, fomp_params, tc_updates=tc_updates)
    
    # Initialisierung der Tracking-Variablen
    dsm_details = {}
    dsm_processes = set(dsm_params.keys())
    fomp_processes = set(fomp_params.keys())
    special_processes = dsm_processes.union(fomp_processes)
    dsm_processes_run = {pid: False for pid in dsm_processes}
    fomp_processes_run = {pid: False for pid in fomp_processes}

    # Der eigentliche Solver-Loop (bleibt unverändert zum letzten Fix)
    for i in range(15):
        something_changed_in_main_loop = False
        while True:
            something_changed_in_tc_loop = False
            for flow in mfa_system.FlowDict.values():
                if np.any(flow.Values != 0) or flow.P_Start in special_processes:
                    continue
                param_name = f"TC_{'_'.join(flow.Name.split('_')[1:3])}"
                if param_name in mfa_system.ParameterDict:
                    input_flows = [f for f in mfa_system.FlowDict.values() if f.P_End == flow.P_Start]
                    if input_flows and all(np.any(f.Values != 0) or f.P_Start == 0 for f in input_flows):
                        total_inflow_values = sum(f.Values for f in input_flows)
                        tc_value = mfa_system.ParameterDict[param_name].Values
                        flow.Values[:, 0] = total_inflow_values[:, 0] * tc_value
                        for i_elem in range(1, len(mfa_system.Elements)):
                            composition_factor = np.divide(
                                total_inflow_values[:, i_elem], total_inflow_values[:, 0], 
                                out=np.zeros_like(total_inflow_values[:, 0]), where=total_inflow_values[:, 0] != 0)
                            flow.Values[:, i_elem] = flow.Values[:, 0] * composition_factor
                        something_changed_in_tc_loop = True
                        something_changed_in_main_loop = True
            if not something_changed_in_tc_loop:
                break
        
        if RUN_DSM_CALCULATION:
            for process_id in dsm_processes:
                if not dsm_processes_run[process_id]:
                    inflows_to_dsm = [f for f in mfa_system.FlowDict.values() if f.P_End == process_id]
                    if inflows_to_dsm and all(np.any(f.Values != 0) for f in inflows_to_dsm):
                        mfa_system, dsm_details_single_run = calculate_dynamic_stock(mfa_system, {process_id: dsm_params[process_id]})
                        dsm_details.update(dsm_details_single_run)
                        dsm_processes_run[process_id] = True
                        something_changed_in_main_loop = True

        if RUN_FOMP_CALCULATION:
            for process_id in fomp_processes:
                 if not fomp_processes_run[process_id]:
                    inflows_to_fomp = [f for f in mfa_system.FlowDict.values() if f.P_End == process_id]
                    if inflows_to_fomp and all(np.any(f.Values != 0) for f in inflows_to_fomp):
                        mfa_system = calculate_fomp(mfa_system, {process_id: fomp_params[process_id]})
                        fomp_processes_run[process_id] = True
                        something_changed_in_main_loop = True

        if not something_changed_in_main_loop and i > 0:
            break
    
    mfa_system = calculate_final_balances(mfa_system)
    return mfa_system, dsm_details

In [20]:
# ===================================================================
# NEUE VISUALISIERUNGSFUNKTIONEN FÜR MONTE CARLO
# ===================================================================

def plot_mc_distribution(df_results, column_name, unit='Mg'):
    """
    Creates an interactive histogram to show the distribution of a key output variable
    from the Monte Carlo simulation, including mean and confidence intervals.
    """
    import plotly.graph_objects as go

    mean_val = df_results[column_name].mean()
    p5 = df_results[column_name].quantile(0.05)
    p95 = df_results[column_name].quantile(0.95)

    fig = go.Figure()
    fig.add_trace(go.Histogram(x=df_results[column_name], name='Distribution', nbinsx=50))
    
    fig.add_vline(x=mean_val, line_width=3, line_dash="dash", line_color="red",
                  annotation_text=f"Mean: {mean_val:.2f} {unit}", annotation_position="top right")
    fig.add_vline(x=p5, line_width=2, line_dash="dot", line_color="green",
                  annotation_text=f"5th Percentile: {p5:.2f}", annotation_position="top left")
    fig.add_vline(x=p95, line_width=2, line_dash="dot", line_color="green",
                  annotation_text=f"95th Percentile: {p95:.2f}")

    fig.update_layout(
        title_text=f'Distribution of "{column_name}"',
        xaxis_title_text=f'Value in {unit}',
        yaxis_title_text='Frequency (Number of Runs)',
        legend_title_text='Metrics'
    )
    fig.show()


def plot_mc_sensitivity_scatter(df_results, input_param_name, output_param_name, unit='Mg'):
    """
    Creates a scatter plot to visualize the relationship between an uncertain
    input parameter and a key output variable. Includes a trendline.
    """
    import plotly.express as px

    if input_param_name not in df_results.columns:
        print(f"ERROR for scatter plot: Input parameter '{input_param_name}' was not stored in the results.")
        return
        
    fig = px.scatter(df_results, x=input_param_name, y=output_param_name,
                     title=f'Sensitivity of "{output_param_name}" to "{input_param_name}"',
                     labels={
                         input_param_name: f'Sampled Value of {input_param_name}',
                         output_param_name: f'Result for {output_param_name} [{unit}]'
                     },
                     trendline="ols", # Adds an ordinary least squares regression trendline
                     trendline_color_override="red"
                    )
    fig.show()

# Section 3: Main workflow (execution)

In [21]:
# ===================================================================
# Section 3: Main Execution (Final Version with Monte Carlo)
# ===================================================================

# --- System-Setup (bleibt gleich) ---
model_classification, index_table = define_model_scope(START_YEAR, END_YEAR, ELEMENTS)
my_mfa_system_base = initialize_mfa_system(model_classification, index_table)
my_mfa_system_base, all_excel_data = load_and_define_processes(my_mfa_system_base, EXCEL_FILE_PATH)

DSM_PARAMS = load_dsm_parameters(all_excel_data)
FOMP_PARAMS = load_fomp_parameters(all_excel_data)
UNCERTAINTY_PARAMS = load_uncertainty_definitions(all_excel_data)

# Die Konfiguration wird nicht mehr hier aufgerufen, sondern innerhalb des Loops
# my_mfa_system_configured, _ = define_flows_and_parameters(my_mfa_system_base, all_excel_data, DSM_PARAMS, FOMP_PARAMS)

# --- Initialisiere Ergebnis-Variablen ---
df_mc_results = None
my_mfa_system_with_results = None
dsm_details = None

# --- ENTSCHEIDUNG UND BERECHNUNG ---
if RUN_MONTE_CARLO:
    print(f"\n--- STARTING MONTE CARLO SIMULATION ({MC_ITERATIONS} iterations) ---")
    mc_run_results = []
    
    # Fortschrittsbalken für eine bessere Übersicht
    try:
        from tqdm.notebook import tqdm
        iterator = tqdm(range(MC_ITERATIONS), desc='MC Runs')
    except ImportError:
        iterator = range(MC_ITERATIONS)

    for i in iterator:
        # 1. Neue Parameterwerte für diese Iteration sampeln
        sampled_values = sample_parameters(UNCERTAINTY_PARAMS)
        
        # 2. Temporäre Kopien der Parameter-Sets erstellen, um die Originale nicht zu verändern
        temp_dsm_params = copy.deepcopy(DSM_PARAMS)
        temp_fomp_params = copy.deepcopy(FOMP_PARAMS)
        tc_updates = {}

        # 3. Gesampelte Werte den richtigen Parameter-Typen zuordnen
        for name, value in sampled_values.items():
            if name.startswith('TC_'):
                tc_updates[name] = value
            elif name.startswith('fomp_'):
                try:
                    parts = name.split('_'); pid = int(parts[1]); param_key = parts[2]
                    if pid in temp_fomp_params: temp_fomp_params[pid][param_key] = value
                except (IndexError, ValueError): pass
            elif name.startswith('dsm_'):
                try:
                    parts = name.split('_'); pid = int(parts[1]); dict_key = parts[2]; param_key = parts[3]; list_index = int(parts[4])
                    if pid in temp_dsm_params and list_index < len(temp_dsm_params[pid][dict_key][param_key]):
                        temp_dsm_params[pid][dict_key][param_key][list_index] = value
                except (IndexError, ValueError): pass

        # 4. MFA-Berechnung mit den für diesen Lauf modifizierten Parametern ausführen
        run_results, _ = run_mfa_calculation(temp_dsm_params, temp_fomp_params, tc_updates=tc_updates)
        
        # 5. Schlüsselergebnisse (KPIs) extrahieren und speichern
        if run_results:
            # Beispiel-KPI: Finaler C-Lagerbestand im Boden (Prozess 8, Element 'CC')
            final_c_stock_soil = run_results.StockDict['S_8'].Values[-1, 3] # S_8, letztes Jahr, 4. Element ('CC')
            
            current_run_data = sampled_values.copy()
            current_run_data['run_id'] = i
            current_run_data['final_C_stock_soil'] = final_c_stock_soil
            mc_run_results.append(current_run_data)

    # 6. Alle Ergebnisse in einem DataFrame zusammenfassen
    df_mc_results = pd.DataFrame(mc_run_results)
    my_mfa_system_with_results = None
    print("\n--- MONTE CARLO SIMULATION COMPLETE ---")

else:
    # --- DETERMINISTISCHER EINZELLAUF ---
    print("\n--- STARTING SINGLE DETERMINISTIC RUN ---")
    my_mfa_system_with_results, dsm_details = run_mfa_calculation(DSM_PARAMS, FOMP_PARAMS)
    print("\nCalculation complete.")

--> Model scope and classifications defined.
--> MFA system object initialized.
--> Defining process and stock structures...


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Loading DSM parameters from sheet '3_1_Definition_DSM'...
--> Successfully loaded configurations for 2 DSM process(es).
--> Loading FOMP parameters from sheet '3_2_Definition_FOMP'...
--> Successfully loaded configurations for 1 FOMP process(es).
--> Loading uncertainty definitions from sheet '4_1_Uncertainty_Parameters'...
--> Successfully loaded 4 uncertainty parameter definition(s).

--- STARTING SINGLE DETERMINISTIC RUN ---
--> MFA system object initialized.
--> Defining process and stock structures...


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Defining flows, parameters, and setting all initial values...
--> All stocks and flows initialized to zero.
--> Populated data for primary input flows.
--> Generating dynamic TC time series via interpolation...
--> Generated 12 dynamic TC parameter(s).
--> Defined 56 parameters in total.
--> Calculating FOMP outflows...
    ... calculating outflow from FOMP process 8
--> FOMP outflow calculation finished.
--> Calculating final stock balances for ALL processes...
--> Stock balance calculation finished.

Calculation complete.


### Überprüfung der Massenbilanz

Die folgende Grafik ist das wichtigste Werkzeug zur Überprüfung der Modellkonsistenz. Sie zeigt den Massenbilanzfehler für jeden Prozess, berechnet nach der Formel:

**`Fehler = Σ Zuflüsse - Σ Abflüsse - Lagerveränderung (dS)`**

**Wie man die Grafik liest:**
* **Perfektes Gleichgewicht:** Ein Prozess ist perfekt bilanziert, wenn sein Balken genau auf der Nulllinie liegt.
* **Positiver Fehler (Balken > 0):** Es wurde mehr Masse "erschaffen" als im System sein dürfte. Mögliche Ursache: Ein Abfluss oder eine Lagerbildung fehlt in der Definition.
* **Negativer Fehler (Balken < 0):** Es ist Masse "verschwunden". Mögliche Ursache: Ein Zufluss fehlt oder ein Abfluss wird doppelt gezählt.

Je größer die Abweichung von Null, desto gravierender ist der Fehler in der Modelllogik für diesen Prozess.

# Section 4: Results and visualization

In [22]:
# ===================================================================
# Section 4 - DEBUG PROBE: Inspecting the final plot data
# ===================================================================

print("\n" + "="*40)
print("FINAL DEBUG PROBE: Inspecting the 'dsm_details' dictionary before plotting")
print("="*40)

if 'dsm_details' in locals() and dsm_details is not None:
    # Prozess 7 ist unser Testfall mit einem Initial Stock von 2000 Mg
    process_id_to_check = 7
    if process_id_to_check in dsm_details:
        details_for_p7 = dsm_details[process_id_to_check]
        
        print(f"\n--- Data for Process {process_id_to_check} ---")
        
        # Wir prüfen das Array, das für den Plot des Initial Stocks verwendet wird
        initial_stock_plot_data = details_for_p7.get('initial_stock_ts')
        
        if initial_stock_plot_data is not None:
            print(f"Shape of 'initial_stock_ts' array: {initial_stock_plot_data.shape}")
            print(f"First 3 values of the 'material' component for the plot: {initial_stock_plot_data[:3, 0]}")
            
            if np.sum(initial_stock_plot_data) == 0:
                print("\n>>> DIAGNOSE: Die an den Plot übergebenen Daten sind komplett Null. Der Fehler liegt in der BERECHNUNGS-Logik.")
            else:
                print("\n>>> DIAGNOSE: Die an den Plot übergebenen Daten enthalten Werte > 0. Der Fehler liegt in der VISUALISIERUNGS-Logik.")
        else:
            print("FEHLER: Der Schlüssel 'initial_stock_ts' wurde in dsm_details für Prozess 7 nicht gefunden.")
    else:
        print(f"FEHLER: Prozess {process_id_to_check} wurde nicht im dsm_details-Dictionary gefunden.")
else:
    print("FEHLER: Die Variable 'dsm_details' existiert nicht oder ist leer. Es wurden keine DSM-Ergebnisse zurückgegeben.")

print("="*40)


FINAL DEBUG PROBE: Inspecting the 'dsm_details' dictionary before plotting

--- Data for Process 7 ---
Shape of 'initial_stock_ts' array: (26, 4)
First 3 values of the 'material' component for the plot: [2000.         1936.84210526 1875.67867036]

>>> DIAGNOSE: Die an den Plot übergebenen Daten enthalten Werte > 0. Der Fehler liegt in der VISUALISIERUNGS-Logik.


# ====================================================================
# Section 4: Results & Visualization
# ====================================================================

### Überprüfung der Massenbilanz

Die folgende Grafik ist das wichtigste Werkzeug zur Überprüfung der Modellkonsistenz. Sie zeigt den Massenbilanzfehler für jeden Prozess, berechnet nach der Formel:

**`Fehler = Σ Zuflüsse - Σ Abflüsse - Lagerveränderung (dS)`**

**Wie man die Grafik liest:**
* **Perfektes Gleichgewicht:** Ein Prozess ist perfekt bilanziert, wenn sein Balken genau auf der Nulllinie liegt.
* **Positiver Fehler (Balken > 0):** Es wurde mehr Masse "erschaffen" als im System sein dürfte. Mögliche Ursache: Ein Abfluss oder eine Lagerbildung fehlt in der Definition.
* **Negativer Fehler (Balken < 0):** Es ist Masse "verschwunden". Mögliche Ursache: Ein Zufluss fehlt oder ein Abfluss wird doppelt gezählt.

In [23]:
# --- Helper Function 4.1: Create an interactive Sankey plot with a threshold slider ---
def plot_interactive_sankey(mfa_system_results):
    """
    Generates an interactive Sankey diagram with widgets to select the year, 
    element, processes, and a value threshold to hide minor flows.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, IntSlider, Dropdown, SelectMultiple, FloatSlider

    all_process_names = [p.Name for p in mfa_system_results.ProcessList]
    all_flows = list(mfa_system_results.FlowDict.values())
    time_items = mfa_system_results.IndexTable.Classification['Time'].Items
    element_items = mfa_system_results.Elements
    
    # Determine a reasonable max for the slider based on calculated flows
    max_flow_value = max(f.Values.max() for f in all_flows if f.Values is not None) if all_flows else 1

    # Create the FigureWidget with an initial, empty Sankey trace
    fig = go.FigureWidget(data=[go.Sankey(node=dict(label=[]), link=dict(source=[], target=[], value=[]))])

    def update_sankey(year, element, processes_to_show, min_flow_value):
        if not processes_to_show:
            with fig.batch_update(): fig.data[0].node.label = []
            return

        label_map = {p.ID: i for i, p in enumerate(mfa_system_results.ProcessList) if p.Name in processes_to_show}
        filtered_labels = list(processes_to_show)
        
        year_index = time_items.index(year)
        element_index = element_items.index(element)

        candidate_flows = [f for f in all_flows if f.P_Start in label_map and f.P_End in label_map]
        
        # NEU: Filter flows based on the slider's threshold value
        final_flows = [f for f in candidate_flows if f.Values[year_index, element_index] >= min_flow_value]

        with fig.batch_update():
            if not final_flows:
                # If no flows are left after filtering, show nodes but no links
                fig.data[0].node.label = filtered_labels
                fig.data[0].link.source, fig.data[0].link.target, fig.data[0].link.value = [], [], []
            else:
                # Update all properties of the Sankey trace
                fig.data[0].node.label = filtered_labels
                fig.data[0].node.color = "blue"
                fig.data[0].link.source = [label_map[f.P_Start] for f in final_flows]
                fig.data[0].link.target = [label_map[f.P_End] for f in final_flows]
                fig.data[0].link.value = [f.Values[year_index, element_index] for f in final_flows]
            
            # Update layout title
            fig.update_layout(title_text=f"MFA Sankey for {element.upper()} in {year} (Flows > {min_flow_value:.2f} Mg)", 
                              font_size=12, height=700, margin=dict(l=10, r=10, b=20, t=50))

    # Create widgets, including the new FloatSlider
    year_slider = IntSlider(min=time_items[0], max=time_items[-1], step=1, value=time_items[0], description='Year')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element')
    process_selector = SelectMultiple(options=all_process_names, value=list(all_process_names), description='Processes', rows=8)
    threshold_slider = FloatSlider(min=0, max=max_flow_value, step=max_flow_value/100, value=0, 
                                   description='Min Flow', continuous_update=False, readout_format='.2f')
    
    interact(update_sankey, year=year_slider, element=element_dropdown, processes_to_show=process_selector, min_flow_value=threshold_slider)
    display(fig)


In [24]:
# --- Helper Function 4.3: Plot Inflow, Stock, and Outflow (Final Corrected Version) ---
def plot_process_dynamics(mfa_system_results, process_definitions):
    """
    Creates three side-by-side line charts showing the dynamics of 
    Inflow, Stock, and Outflow, using process type metadata for smarter titles.
    """
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown

    # <<< HIER IST DIE ANPASSUNG: Der korrekte Spaltenname aus deiner Excel-Datei >>>
    PROCESS_TYPE_COLUMN_NAME = 'Process_Type' 

    # Check if the column exists to avoid errors
    has_type_column = PROCESS_TYPE_COLUMN_NAME in process_definitions.columns
    if not has_type_column:
        print(f"Warning: Column '{PROCESS_TYPE_COLUMN_NAME}' not found in '2_1_Definition_Processes'. Smart titles will be disabled.")

    process_options = {p.Name: p.ID for p in mfa_system_results.ProcessList if f"S_{p.ID}" in mfa_system_results.StockDict}
    if not process_options:
        print("No processes with stocks found to plot.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    fig = go.FigureWidget(make_subplots(rows=1, cols=3, subplot_titles=("Inflow", "Stock (S)", "Outflow")))

    def update_plot(process_name, element):
        pid = process_options[process_name]
        element_index = element_items.index(element)

        inflows = [f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_End == pid]
        inflow_ts = sum(inflows) if inflows else np.zeros(len(time_axis))
        stock_ts = mfa_system_results.StockDict[f'S_{pid}'].Values[:, element_index]
        outflows = [f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_Start == pid]
        outflow_ts = sum(outflows) if outflows else np.zeros(len(time_axis))
        
        subplot_titles = (f"Inflow to '{process_name}'", f"Stock in '{process_name}'", f"Outflow from '{process_name}'")
        
        if has_type_column:
            process_type = process_definitions.loc[process_definitions['ID'] == pid, PROCESS_TYPE_COLUMN_NAME].iloc[0]
            if process_type == 'Input':
                subplot_titles = ("Primary System Input", f"Stock in '{process_name}'", f"Outflow from '{process_name}'")
            elif process_type == 'Output':
                subplot_titles = (f"Inflow to '{process_name}'", f"Stock in '{process_name}'", "Final System Output (Sink)")

        with fig.batch_update():
            fig.data, fig.layout.annotations = [], []
            fig.add_trace(go.Scatter(x=time_axis, y=inflow_ts, mode='lines', name='Inflow'), row=1, col=1)
            fig.add_trace(go.Scatter(x=time_axis, y=stock_ts, mode='lines', name='Stock'), row=1, col=2)
            fig.add_trace(go.Scatter(x=time_axis, y=outflow_ts, mode='lines', name='Outflow'), row=1, col=3)
            
            fig.layout.annotations = [
                dict(x=0.155, y=1.05, text=subplot_titles[0], showarrow=False, xref='paper', yref='paper', xanchor='center'),
                dict(x=0.5, y=1.05, text=subplot_titles[1], showarrow=False, xref='paper', yref='paper', xanchor='center'),
                dict(x=0.845, y=1.05, text=subplot_titles[2], showarrow=False, xref='paper', yref='paper', xanchor='center')
            ]
            fig.update_layout(title=f"Dynamics for Process: '{process_name}' | Element: {element.upper()}", height=400, showlegend=False)
            fig.update_xaxes(title_text="Year")
            fig.update_yaxes(title_text="Mass [Mg]", row=1, col=1)

    process_dropdown = Dropdown(options=list(process_options.keys()), description='Process:')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, process_name=process_dropdown, element=element_dropdown)
    display(fig)

In [25]:
# --- Helper Function 4.4: Plot dynamic stock composition (Final Interactive Version) ---
def plot_dynamic_stock_composition(dsm_details, mfa_system_results):
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown, Checkbox

    process_options = list(dsm_details.keys())
    if not process_options: return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    fig = go.FigureWidget()

    def update_plot(process_id, element, show_as_bars):
        details = dsm_details.get(process_id, {})
        element_index = element_items.index(element)
        
        # Get data from the details dictionary
        initial_stock_ts_all_elements = details.get('initial_stock_ts', np.zeros((len(time_axis), len(element_items))))
        inflow_stocks_material = details.get('inflow_stock_ts_by_cat', [])
        category_names = details.get('category_names', [])
        mean_lifetimes = details.get('mean_lifetimes', [])
        
        # Get composition of the mixed inflow for the new stock parts
        inflows = [f.Values for f in mfa_system_results.FlowDict.values() if f.P_End == process_id]
        total_inflow_values = sum(inflows) if inflows else np.zeros((len(time_axis), len(element_items)))
        inflow_comp_factor = np.divide(total_inflow_values[:, element_index], total_inflow_values[:, 0], out=np.zeros(len(time_axis)), where=total_inflow_values[:, 0]!=0)

        with fig.batch_update():
            fig.data = []
            chart_type = go.Bar if show_as_bars else go.Scatter
            
            # --- Plot 1: The decaying initial stock ---
            initial_stock_ts_element = initial_stock_ts_all_elements[:, element_index]
            fig.add_trace(chart_type(x=time_axis, y=initial_stock_ts_element, name='Initial Stock (Decaying)', hoverinfo='x+y',
                                     **({'mode':'lines', 'line':dict(width=0.5), 'stackgroup':'one'} if not show_as_bars else {})))

            # --- Plot 2: The stock from new inflows, category by category ---
            for i, stock_ts_material in enumerate(inflow_stocks_material):
                stock_ts_element = stock_ts_material * inflow_comp_factor
                label = f"{category_names[i]} ({mean_lifetimes[i]} yrs)"
                fig.add_trace(chart_type(x=time_axis, y=stock_ts_element, name=label, hoverinfo='x+y',
                                         **({'mode':'lines', 'line':dict(width=0.5), 'stackgroup':'one'} if not show_as_bars else {})))

            process_name = next((p.Name for p in mfa_system_results.ProcessList if p.ID == process_id), "")
            fig.update_layout(barmode='stack' if show_as_bars else None, title=f"Dynamic Stock Composition for Process: '{process_name}' ({element.upper()})",
                              xaxis_title="Year", yaxis_title=f"Stock in Mg")

    process_dropdown = Dropdown(options=process_options, description='Process:')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    chart_type_checkbox = Checkbox(value=False, description='Show as Bar Chart')
    
    interact(update_plot, process_id=process_dropdown, element=element_dropdown, show_as_bars=chart_type_checkbox)
    display(fig)

In [26]:
# --- Helper Function 4.5: Plot the dynamics of a FOMP process ---
def plot_fomp_dynamics(mfa_system_results, fomp_params_config):
    """
    Creates side-by-side line charts for Inflow, Stock, and Outflow
    for a process calculated with FOMP. Includes interactive widgets.
    """
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown

    # Create a mapping of process names to IDs for the dropdown, only for FOMP processes
    process_options = {
        p.Name: p.ID 
        for p in mfa_system_results.ProcessList 
        if p.ID in fomp_params_config
    }
    if not process_options:
        print("No processes with FOMP parameters are defined in the configuration.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    fig = go.FigureWidget(make_subplots(rows=1, cols=3, subplot_titles=("Total Inflow", "Absolute Stock (S)", "Outflow (Mineralization)")))

    def update_plot(process_name, element):
        pid = process_options[process_name]
        element_index = element_items.index(element)

        # Get the time series data for the selected process
        inflow_ts = sum(f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_End == pid)
        stock_ts = mfa_system_results.StockDict.get(f'S_{pid}').Values[:, element_index]
        outflow_ts = sum(f.Values[:, element_index] for f in mfa_system_results.FlowDict.values() if f.P_Start == pid)
        
        with fig.batch_update():
            fig.data = [] # Clear existing data
            fig.add_trace(go.Scatter(x=time_axis, y=inflow_ts, mode='lines', name='Inflow'), row=1, col=1)
            fig.add_trace(go.Scatter(x=time_axis, y=stock_ts, mode='lines', name='Stock'), row=1, col=2)
            fig.add_trace(go.Scatter(x=time_axis, y=outflow_ts, mode='lines', name='Outflow'), row=1, col=3)
            
            title_text = f"FOMP Dynamics for Process: '{process_name}' | Element: {element.upper()}"
            fig.update_layout(title_text=title_text, height=400, showlegend=False)
            fig.update_xaxes(title_text="Year")
            fig.update_yaxes(title_text="Mass [Mg]", row=1, col=1)

    # Create widgets for interaction
    process_dropdown = Dropdown(options=list(process_options.keys()), description='Process:')
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    
    interact(update_plot, process_name=process_dropdown, element=element_dropdown)
    display(fig)

In [27]:
# --- Helper Function 4.6: Plot the dynamics of selected flows over time ---
def plot_flow_dynamics(mfa_system_results):
    """
    Creates an interactive line/bar chart to show the development of selected
    flows over time for a chosen element.
    """
    import plotly.graph_objects as go
    from ipywidgets import interact, Dropdown, SelectMultiple, Checkbox

    # Create options for the widgets
    flow_options = sorted(list(mfa_system_results.FlowDict.keys()))
    if not flow_options:
        print("No flows found in the system to plot.")
        return
        
    element_items = mfa_system_results.Elements
    time_axis = mfa_system_results.IndexTable.Classification['Time'].Items
    
    # Use FigureWidget for efficient updates
    fig = go.FigureWidget()

    def update_plot(flows_to_show, element, show_as_bars):
        # Use batch_update for smooth interaction
        with fig.batch_update():
            fig.data = [] # Clear previous traces
            if not flows_to_show:
                fig.update_layout(title_text="Please select one or more flows to display.")
                return

            element_index = element_items.index(element)
            chart_type = go.Bar if show_as_bars else go.Scatter

            # Add a trace for each selected flow
            for flow_id in flows_to_show:
                flow_obj = mfa_system_results.FlowDict.get(flow_id)
                if flow_obj:
                    trace_props = dict(x=time_axis, y=flow_obj.Values[:, element_index], name=flow_id)
                    if not show_as_bars:
                        trace_props.update(mode='lines')
                    fig.add_trace(chart_type(**trace_props))
            
            # Update layout and title
            fig.update_layout(
                barmode='stack' if show_as_bars else 'overlay',
                title=f"Time Series for Selected Flows ({element.upper()})",
                xaxis_title="Year",
                yaxis_title="Mass in Mg",
                hovermode="x unified"
            )

    # Create widgets
    flow_selector = SelectMultiple(options=flow_options, value=[flow_options[0]] if flow_options else [], description='Flows:', rows=10)
    element_dropdown = Dropdown(options=element_items, value=element_items[0], description='Element:')
    chart_type_checkbox = Checkbox(value=False, description='Show as Bar Chart')

    interact(update_plot, flows_to_show=flow_selector, element=element_dropdown, show_as_bars=chart_type_checkbox)
    display(fig)

In [28]:
# ===================================================================
# Section 4: Results and Visualization (Final Corrected Version)
# ===================================================================

# Prüfe zuerst, welcher Modus gelaufen ist
if RUN_MONTE_CARLO:
    # --- VISUALIZATION FOR MONTE CARLO RUN ---
    # Stelle sicher, dass Ergebnisse vorhanden sind
    if df_mc_results is not None and not df_mc_results.empty:
        print("\n--- Monte Carlo Results Summary ---")
        print(df_mc_results.describe())

        # Zeige die Verteilung des wichtigsten Ergebnisses
        plot_mc_distribution(df_mc_results, 'final_C_stock_soil', unit='Mg C')

        # Zeige eine Beispiel-Sensitivitätsanalyse
        uncertain_input_to_test = 'fomp_8_k1'
        if uncertain_input_to_test in df_mc_results.columns:
            plot_mc_sensitivity_scatter(df_mc_results, uncertain_input_to_test, 'final_C_stock_soil', unit='Mg C')
        else:
            print(f"\nNOTE: Sensitivity plot for '{uncertain_input_to_test}' skipped, as it was not in the uncertainty parameters.")
    else:
        print("\nINFO: Monte Carlo run was selected, but no results were generated.")
        
else:
    # --- VISUALIZATION FOR SINGLE DETERMINISTIC RUN ---
    # Stelle sicher, dass Ergebnisse vorhanden sind
    if my_mfa_system_with_results is not None:
        print("\n--- Displaying results for single run ---")
        
        # --- 4.1 Interactive Sankey Diagram ---
        print("Displaying interactive Sankey diagram:")
        # Dieser Aufruf ist jetzt sicher, da er nur im richtigen "Pfad" stattfindet
        plot_interactive_sankey(my_mfa_system_with_results)
        
        # --- 4.2 Process Dynamics Plot (Inflow-Stock-Outflow) ---
        print("\nDisplaying Inflow-Stock-Outflow Dynamics:")
        plot_process_dynamics(my_mfa_system_with_results, all_excel_data['2_1_Definition_Processes'])
        
        # --- 4.3 Dynamic Stock Composition Plot ---
        print("\nDisplaying Dynamic Stock Composition:")
        if dsm_details:
             plot_dynamic_stock_composition(dsm_details, my_mfa_system_with_results)
        else:
             print("--> DSM calculation was not run or no DSM processes are defined. Nothing to display.")
        
        # --- 4.4 FOMP Dynamics Plot ---
        print("\nDisplaying FOMP Dynamics:")
        if FOMP_PARAMS:
             plot_fomp_dynamics(my_mfa_system_with_results, FOMP_PARAMS)
        else:
             print("--> No FOMP parameters defined. Nothing to display.")
        
        # --- 4.5 Flow Dynamics Plot ---
        print("\nDisplaying Flow Dynamics:")
        plot_flow_dynamics(my_mfa_system_with_results)
    else:
        print("\nINFO: Single run was selected, but no results were generated.")


--- Displaying results for single run ---
Displaying interactive Sankey diagram:


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'link': {'source': [0, 1, 2, 3, 3, 4, 4, 5, 6, 7, 7, 8, 9, 9],
                       'target': [2, 2, 3, 4, 5, 0, 1, 6, 9, 0, 1, 0, 8, 7],
                       'value': [100.0, 100.0, 200.0, 100.0, 100.0, 50.0, 50.0,
                                 100.0, 53.414960640068124, 63.19769925769572, 0.0,
                                 30.792318331808985, 21.36598425602725,
                                 32.04897638404087]},
              'node': {'color': 'blue',
                       'label': [Atmosphere, Environment, Cultivation, Harvest,
                                 Food, Straw d&C, Use_Straw_Roof,
                                 Incineration_Roof, Composting_roof, Roof_EoL]},
              'type': 'sankey',
              'uid': '2879add0-803d-4da9-9213-0cbb92e38b85'}],
    'layout': {'font': {'size': 12},
               'height': 700,
               'margin': {'b': 20, 'l': 10, 'r': 10, 't': 50},
               'template': '...',
               


Displaying Inflow-Stock-Outflow Dynamics:


interactive(children=(Dropdown(description='Process:', options=('Atmosphere', 'Environment', 'Use_Straw_Roof',…

...

interactive(children=(Dropdown(description='Process:', options=('Composting_roof',), value='Composting_roof'),…

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'Inflow',
              'type': 'scatter',
              'uid': '97d13237-09ca-4250-a765-8adab6b22d4a',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'xaxis': 'x',
              'y': array([21.36598426, 20.24489727, 19.19070299, 18.20230739, 17.27954307,
                          16.42344676, 15.63675735, 14.92924059, 14.38450652, 14.40342197,
                          15.43152529, 16.72023112, 17.72164287, 18.66641031, 20.01082727,
                          21.80306498, 23.94457433, 26.26300572, 28.66266014, 30.96854021,
                          32.39082319, 32.5362709 , 33.6454829 , 35.88864593, 36.97399669,
                          37.2104085 ]),
              'yaxis': 'y'},
             {'mode': 'lines',
              'name': 'Stoc


Displaying Flow Dynamics:


interactive(children=(SelectMultiple(description='Flows:', index=(0,), options=('F_00_02', 'F_01_02', 'F_02_03…

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'F_00_02',
              'type': 'scatter',
              'uid': '0b774253-caa1-42a6-a7ac-3247cba4c2e2',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': array([100., 110., 120., 130., 140., 150., 160., 170., 180., 190., 200.,  10.,
                          220., 230., 240., 250., 260., 270., 280., 290., 300., 310., 320., 330.,
                          340., 350.])}],
    'layout': {'barmode': 'overlay',
               'hovermode': 'x unified',
               'template': '...',
               'title': {'text': 'Time Series for Selected Flows (MATERIAL)'},
               'xaxis': {'title': {'text': 'Year'}},
               'yaxis': {'title': {'text': 'Mass in Mg'}}}
})

# Section 5: Export Results

In [29]:
# ====================================================================
# Section 5: Export Results
# ====================================================================

def export_results_to_excel(mfa_system_results, output_filename="mfa_results.xlsx"):
    """
    Exports all calculated flows and stocks into a single Excel file with multiple sheets.
    """
    print(f"\n--> Exporting results to '{output_filename}'...")
    
    time_index = mfa_system_results.IndexTable.Classification['Time'].Items
    elements = mfa_system_results.Elements
    
    with pd.ExcelWriter(output_filename) as writer:
        # --- Export Flows ---
        flow_data_rows = []
        for name, flow_obj in mfa_system_results.FlowDict.items():
            for i, year in enumerate(time_index):
                row = {'Flow_ID': name, 'Year': year}
                for j, element in enumerate(elements):
                    row[element] = flow_obj.Values[i, j]
                flow_data_rows.append(row)
        df_flows = pd.DataFrame(flow_data_rows)
        df_flows.to_excel(writer, sheet_name='Flows_ts', index=False)
        
        # --- Export Stocks ---
        stock_data_rows = []
        for name, stock_obj in mfa_system_results.StockDict.items():
            for i, year in enumerate(time_index):
                row = {'Stock_ID': name, 'Year': year}
                for j, element in enumerate(elements):
                    row[element] = stock_obj.Values[i, j]
                stock_data_rows.append(row)
        df_stocks = pd.DataFrame(stock_data_rows)
        df_stocks.to_excel(writer, sheet_name='Stocks_ts', index=False)
        
    print("--> Export complete.")

In [30]:
# ====================================================================
# Section 5: Export Results
# ====================================================================

# Call the export function
export_results_to_excel(my_mfa_system_with_results, output_filename="rye_mfa_results_v1.xlsx")


--> Exporting results to 'rye_mfa_results_v1.xlsx'...
--> Export complete.


<img src="system_flow_diagram.svg" alt="system_flow_diagram">

## 0 Load packages

This cell imports all necessary packages. It also loads the ODYM framework and the bioDYM_addon, both are included as files in the project (see folder /framework). 

## 3 MFA Calculations

Now, the solution of the MFA is calculated. Since most flows have either input data or TCs and substance contents are given, they can be easily calculated. However, this system includes a dynamic stock modeling (dsm) and a first order model process (FOMP) for the mineralization of carbon in soil. The idea is that first, all flows are calculated with TCs that are independent of dsm or FOMP. Then, dsm is performed and subsequently, all flows up to the FOMP are calculated. After that, the bioDYM_addon functions are used to calculate the mineralization. Finally, all following flows and stocks are calculated.

### 3.1 Solution MFA

### 3.1.1 Solution MFA pt. I (until MBC dynamic stock modelling)


### 3.1.4 Solution MFA pt. IV (FOMP mineralization process)

The carbon mineralization process in the soil is calculated with a first order decay model according to (Cayuela et al., 2010) based on (Robertson & Paul, 2000): 

$$ C_{remaining} (t)=f \cdot exp⁡(-k_{1} \cdot t)+(100\%-f) \cdot exp⁡(-k_{2} \cdot t) $$

To keep calculations simple, it is assumed that this is the only equation that leads to outflows of the process, the remaining fractions of the material accumulate as stock without any emissions. (Cayuela et al., 2010) give parameter values for green waste biochar, they are used here.

\
\
Literature

Cayuela, M. L., Oenema, O., Kuikman, P. J., Bakker, R. R., & Van GROENIGEN, J. W. (2010). Bioenergy by-products as soil amendments? Implications for carbon sequestration and greenhouse gas emissions: C AND N DYNAMICS FROM BIOENERGY BY-PRODUCTS IN SOIL. GCB Bioenergy, no-no. https://doi.org/10.1111/j.1757-1707.2010.01055.x

Robertson, G. P., & Paul, E. (2000). Decomposition and Soil Organic Matter Dynamics. Decomposition and Soil Organic Matter Dynamics., 104–116. https://doi.org/10.1007/978-1-4612-1224-9_8